# Inspecting normalizations

In [1]:
import os
import random

import datasets
import numpy as np
import torch
import transformers
from tokenizers.normalizers import Normalizer
from tokenizers import NormalizedString

/home/bracke/miniconda3/envs/gpu-venv-transnormer/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# number of paragraphs per split (= century)
N = 100

In [3]:
# Fix seeds for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

In [4]:
# Load DTA validation split for three time periods
datadir = "/home/bracke/data/dta/dtak/"
data_files = {
    "1600to1699": os.path.join(datadir, "dta-ps-1600-1699-validation.jsonl"),
    "1700to1799": os.path.join(datadir, "dta-ps-1700-1799-validation.jsonl"),
    "1800to1899": os.path.join(datadir, "dta-ps-1800-1899-validation.jsonl"),
}
ds = datasets.load_dataset("json", data_files=data_files)


ds["1600to1699"] = ds["1600to1699"].shuffle().select(range(N))
ds["1700to1799"] = ds["1700to1799"].shuffle().select(range(N))
ds["1800to1899"] = ds["1800to1899"].shuffle().select(range(N))

ds = ds.rename_column("text", "orig")

Using custom data configuration default-86e8bd6a081066fa


Extracting data files: 100%|██████████| 3/3 [00:00<00:00, 1360.90it/s]


Dataset json downloaded and prepared to /home/bracke/.cache/huggingface/datasets/json/default-86e8bd6a081066fa/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51. Subsequent calls will reuse this data.


100%|██████████| 3/3 [00:00<00:00, 98.24it/s]


In [5]:
# GPU set-up
device = torch.device('cuda:1' if torch.cuda.is_available() else "cpu")

# Load model
checkpoint = "../../models/models_2023-02-28_16-52/model_final"
model = transformers.EncoderDecoderModel.from_pretrained(
    checkpoint, 
    ).to(device)

# Load tokenizer
class CustomNormalizer:
    def normalize(self, normalized: NormalizedString):
        normalized.nfd() # unicode decomposition
        normalized.replace("ſ", "s")
        normalized.replace("ꝛ", "r")
        normalized.replace(chr(0x0303), "") # drop combining tilde
        normalized.replace(chr(0x0364), "e")
        normalized.replace("æ", "ae")
        normalized.replace("ů","ü")
        normalized.replace("Ů","Ü")
        normalized.nfc()

tokenizer_hmbert_custom = transformers.AutoTokenizer.from_pretrained("dbmdz/bert-base-historic-multilingual-cased")
tokenizer_hmbert_custom.backend_tokenizer.normalizer = Normalizer.custom(CustomNormalizer())
tokenizer_bert = transformers.AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

In [6]:
def generate_normalization(batch):
    inputs = tokenizer_hmbert_custom(batch["orig"], padding="max_length", truncation=True, max_length=128, return_tensors="pt")
    input_ids = inputs.input_ids.to(device)
    attention_mask = inputs.attention_mask.to(device)

    outputs = model.generate(input_ids, attention_mask=attention_mask)

    output_str = tokenizer_bert.batch_decode(outputs, skip_special_tokens=True)

    # batch["norm_pred_ids"] = outputs
    batch["norm_pred_str"] = output_str

    return batch


ds = ds.map(
    generate_normalization, 
    batched=True, 
    batch_size=8, 
    )


Parameter 'function'=<function generate_normalization at 0x7fc3ec06a5e0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.
  0%|          | 0/13 [00:00<?, ?ba/s]/home/bracke/miniconda3/envs/gpu-venv-transnormer/lib/python3.9/site-packages/transformers/generation/utils.py:1186: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use a generation configuration file (see https://huggingface.co/docs/transformers/main_classes/text_generation)
 

In [7]:
import pandas as pd
# don't truncate cells with long text
pd.set_option('display.max_colwidth', None)

part = "1600to1699"
df1600to1699 = pd.DataFrame(data={"orig" : ds[part]["orig"], "norm_pred" : ds[part]["norm_pred_str"]})
part = "1700to1799"
df1700to1799 = pd.DataFrame(data={"orig" : ds[part]["orig"], "norm_pred" : ds[part]["norm_pred_str"]})
part = "1800to1899"
df1800to1899 = pd.DataFrame(data={"orig" : ds[part]["orig"], "norm_pred" : ds[part]["norm_pred_str"]})


In [8]:
from transnormer.data.loader import read_leipzig_raw
data_modern = read_leipzig_raw("/home/bracke/code/transnormer/data/raw/leipzig-corpora/deu_news_2020_1M-sentences.txt")
ds_modern = datasets.Dataset.from_dict(data_modern)
ds_modern = ds_modern.shuffle().select(range(N))
ds_modern = ds_modern.map(generate_normalization, batched=True, batch_size=8, )
df_modern = pd.DataFrame(data={"orig" : ds_modern["orig"], "norm_pred" : ds_modern["norm_pred_str"]})

  0%|          | 0/13 [00:00<?, ?ba/s]/home/bracke/miniconda3/envs/gpu-venv-transnormer/lib/python3.9/site-packages/transformers/generation/utils.py:1273: UserWarning: Neither `max_length` nor `max_new_tokens` has been set, `max_length` will default to 128 (`generation_config.max_length`). Controlling `max_length` via the config is deprecated and `max_length` will be removed from the config in v5 of Transformers -- we recommend using `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
100%|██████████| 13/13 [00:27<00:00,  2.13s/ba]


## Inspect dataframes

### 1600 to 1699

In [9]:
df1600to1699.head(20)

,orig,norm_pred
0,Weltliebende ſollen Bußthun 57,Weltliebende sollen Bußtun 57
1,"III. Sententia occidens, der toͤdtende Außſpruch/ heißt nicht leben/ ſondern ewig todt ſeyn/ welcher Tod nach ſich zeucht den Verluſt alles Guten und Verſtoſſung von Gottes Angeſicht/ was aber diß fuͤr ein Qual und Pein ſey/ ewig von GOtt verſtoſſen ſeyn/ das kan kein Creatur außdencken und außdichten/ kein Zung außreden und außſprechen/ der zeitliche Tod wird manchem durchſuͤſſet/ und die Bitterkeit vertrieben/ wo nicht durch Gottes Wort/ als die beſte Medicin, doch durch Hoffnung der Reputation, wie Saul in ſeiner Cron als ein Koͤnig ſterben wollen.","III. Sentimenta occidens, der tötende Außspruch / heißt nicht leben / sondern ewig tot sein / welcher Tod nach sich zeucht den Verlust alles Guten und Verstossung von Gottes Angesicht / was aber dies für ein Qual und Pein sei / ewig von GOT verstoßen sein / das kan kein Kreatur ausdenken und aus demdichten / kein Zung ausreden und aussprechen / der zeitliche Tod wird manchem durchsüsset / und die Bitterkeit vertrieben / wo nicht durchzuschieden."
2,Braucht der Herr nichts?,Braucht der Herr nichts?
3,Vbel reiſen.,Vbel reisen.
4,12.,12.
5,Halt eines Glaſes Rand uͤber ein brennend Liecht/ laß wol haiß werden/ dupff geſchwind mit eim naſſen Finger darauff/ ſo wird das Glaß ein ſchnapp thun/ vnd ein kleines Rißlein bekommen/ daran halt eine brennende Lunden/ blaß daran vnd fuͤhre ſie alſo am Glaß forth wo hin du wilt/ doch nit zu geſchwind/ ſo wird ſich das Glaß alſo zerſchneiden vnd zertheilen laſſen:,"Halt eines Glases Rand über ein brennend Liecht / laß wohl Haiß werden / dupff geschwind mit eim nassen Finger darauff / so wird das Glaß ein schnappe tun / und ein kleines Rißlein bekommen / daran halt eine brennende Lunden / blaß daran und führe sie also am Gloß forth wohin du will du willst / doch nicht zu geschwinden, so wird sich das Glanz also zerschneiden und zerteilen lassen :"
6,VI. 503 . wie unveraͤnderlich?,VI. 500. wie unveränderlich?
7,Und wer auß Gott/ zu Gottes Ehr/ und nach Goͤttlicher Regul/ ſolche hohe Gaben angewendet/ derſelben in der Furcht GOttes/ Einfalt und Lauterkeit ſeines Hertzens/ auß motiv des Glaubens und der Liebe Chriſti/ ohne Welt-Danck und Eigenſucht recht gebraucht/ zieret ſeine Ampts-Gaben mit heiligmachenden Gaben/ iſt maͤchtig λόγῳ κ ἔϱγῳ in Worten und Thaten wie Moſes Act. 7. glaubet auch/ daß Chriſtum lieb haben beſſer ſey/ als alles wiſſen/ der iſt/ wie ein Wildpret/ alſo auch von Gott erkannt 2. Tim. 2/ 19. und hertzlich geliebet/ der wird auch leuchten an jenem Tage/ wie die Sterne am Himmel.,Und wer auß Gott / zu Gottes Ehre / und nach Göttlicher Regul / solche hohe Gaben angewendet / derselben in der Furcht GOttes / Einfalt und Lauterkeit seines Hertzens / Auß motivisch des Glaubens und der Liebe Christi / ohne Welt - Dank und Eigensucht recht gebraucht / zieret seine Ampts - Gaben mit heiligmachenden Gaben / ist mächtig kam in Worten und Taten wie Moses Act. 7. glaubet auch / daß Christus lieb haben besser sei / als alles
8,Diß kan gar wol ſein/ vornemlich im erſten Fruͤhling/ da die heuffig aus der Erden empor ſteigende duͤnſte ſtarcke refractiones geben/ doch nicht allein in den Oſtern/ ſondern auch ſonſten:,Dies kann gar wohl sein / vornehmlich im ersten Frühling / da die heufig aus der Erden empor steigende dünne reflationes geben / doch nicht allein in den Ostern / sondern auch sonsten :
9,Jch habe geſagt/ daß ihr Freunde ſeyd/ dann alles was ich hab von meinem Vater gehoͤret/ hab ich euch kund gethann.,Ich habe gesagt / daß ihr Freunde seid / dann alles was ich habe von meinem Vater gehört / habe ich euch kundgetann.


#### Notes on 1600-1699

General stuff
* Length limitations (not specific to this time period); e.g., ex. 1 and 7 in 1600-1699
* Digits may get replaced wrongly; e.g. ex. 6 in 1600-1699, 

* After including additional training data (ridges, deu_news_2020), no longer
  replaces forward slashes (`/`)
* Training data used for this model was not older than ~1790
* Many errrors where non-modern words or non-words are produced: e.g. "Haiß",
  "dupff", "Gloß forth" (ex. 5), "Regul", "Hertzens", "Ampts" (ex. 7)

### 1700 to 1799

In [10]:
df1700to1799.head(20)

,orig,norm_pred
0,"Sie ſpielte die Fromme, die Sittſame, die Eingezogene, um meine Neugier rege zu machen.","Sie spielte die Fromme, die Sittsame, die Eingezogene, um meine Neugier rege zu machen."
1,"Denn da der Menſchen Lebens-Zeit Ohn’ all’ Empfindlichkeit Ganz unvermerkt von hinnen eilet; So ſcheinet es, daß jeder Augenblick Recht ordentlich dieſelbe teilet, Und ſo zu ſagen uns ein wahres Stuͤck Von unſ’rer Dauer zeiget.","Denn da der Menschen Lebenszeit Unsere Empfindlichkeit Ganz unvermerkt von hinnen eilet ; So scheinet es, daß jeder Augenblick Recht ordentlich dieselbe teilet, Und sozusagen uns ein wahres Stück Von unserer Dauer zeiget."
2,745 num.,74 num.
3,"Leichter laͤßt es ſich glauben, daß man an dem Hodenſakke maͤnnlicher Thiere eine Bewegung wahrgenommen habe . Dieſer ſteiget in der Frucht in die Hoͤhe, und umfaſſet die Hode und deren Scheide.","Leichter läßt es sich glauben, daß man an dem Hodensacke männlicher Tiere eine Bewegung wahrgenommen habe. Dieser steigt in der Frucht in die Höhe, und umfasset die Hode und deren Scheide."
4,"Dieſe Nerven nun ſind es, welche die Eindruͤcke, die die Dinge auf den Menſchen machen, aufnehmen, und dieſelben nach der gemeinſten Meinung vermittelſt des Nervenſafts, einer ſehr feinen, unſichtbaren, aber, wie es ſcheint, mit der elektriſchen und magnetiſchen Materie verwandten Fluͤſſigkeit, bis zum Gehirn fortfuͤhren, von wo ihn die Seele auf eine uns natuͤrlich unbekannte Weiſe empfaͤngt.","Diese Nerven nun sind es, welche die Eindrücke, die die Dinge auf den Menschen machen, aufnehmen, und dieselben nach der gemeinsten Meinung vermittels des Nervenats, einer sehr feinen, unsichtbaren, aber, wie es scheint, mit der elektrischen und magnetischen Materie verwandten Flüssigkeit, bis zum Gehirn fortführen, von wo ihn die Seele auf eine uns natürlich unbekannte Weise empfängt."
5,Goͤtz.,Götz.
6,"So wie die poetiſche Beſchreibungen der Schlachten und Gefechte dem epiſchen Gedicht ein großes Leben geben, ſo ſind ſie auch ein guter Gegenſtand der Mahlerey.","So wie die poetische Beschreibungen der Schlachten und Gefechte dem epischen Gedicht ein großes Leben geben, so sind sie auch ein guter Gegenstand der Malerei."
7,Plin. Hiſt. Nat. L. VII. c. 30.,Plin. Hist. Nat. L. VII. c. 30.
8,"Unter denen uns heute begegneten verſchiedenen Gattungen von Betlern war ein Knabe von etwa 13 Jahren, dem eine hoͤlzerne Maſchine, und uͤber derſelben ein in acht Straͤngen zertheiltes Strik mit einem platten Gloͤkchen an jedem Ende, vom Halſe herabhieng.","Unter denen uns heute begegneten verschiedenen Gattungen von Betlern war ein Knabe von etwa 13 Jahren, dem eine hölzerne Maschine, und über derselben ein in acht Strängen zerteiltes Strick mit einem platten Glöckchen an jedem Ende, vom Halse herabhing."
9,"Die Audienz geſchahe auf eben die Weiſe, als bei den Großen in Jedo.","Die Audienz geschah auf eben die Weise, als bei den Großen in Jedo."


### 1800 to 1899

In [11]:
df1800to1899.head(20)

,orig,norm_pred
0,Die Einwohner flüchteten ſich auf die Inſeln im See und nahmen zu größerer Sicherheit alle Boote am Ufer mit.,Die Einwohner flüchteten sich auf die Inseln im See und nahmen zu größerer Sicherheit alle Boote am Ufer mit.
1,"Die Kugel, die ſich darin heranbewegte, iſt das Ei ſelbſt.","Die Kugel, die sich darin heranbewegte, ist das Ei selbst."
2,"Bei den meiſten Eidechſen, den Schildkröten und Krokodilen tritt hierzu noch die Nickhaut, die ebenfalls eine Knorpelplatte enthält und von dem inneren Augenwinkel her mehr oder minder weit, zuweilen ganz vollſtändig über das Auge herübergezogen werden kann.","Bei den meisten Eidechsen, den Schildkröten und Krokodilen tritt hierzu noch die Nickhaut, die ebenfalls eine Knorpelplatte enthält und von dem inneren Augenwinkel her mehr oder minder weit, zuweilen ganz vollständig über das Auge herübergezogen werden kann."
3,"Effi erholte ſich, nahm um ein Geringes wieder zu (der alte Brieſt gehörte zu den Wiegefanatikern) und verlor ein gut Teil ihrer Reizbarkeit.","Effi erholte sich, nahm um ein Geringes wieder zu ( der alte Briest gehörte zu den Wiegefanatikern ) und verlor ein gut Teil ihrer Reizbarkeit."
4,Alſo lebt Ermelinde und iſt gerettet!,Also lebt Ermelinde und ist gerettet!
5,"In Ländern, die vom vulkaniſchen Feuer unterhöhlt ſind, und in einem Himmelsſtrich, wo die Natur ſo großartig und dabei ſo geheimnisvoll unruhig iſt, ſteigert ſich von ſelbſt die Aufmerkſamkeit auf phyſikaliſche Erſcheinungen, und damit die Neubegier.","In Ländern, die vom vulkanischen Feuer unterhöhlt sind, und in einem Himmelsstrich, wo die Natur so großartig und dabei so geheimnisvoll unruhig ist, steigert sich von selbst die Aufmerksamkeit auf physikalische Erscheinungen, und damit die Neubegier."
6,"Die Wirbelſäule der Vögel läßt ſtets die verſchiedenen Abtheilungen in Hals-, Rücken-, Lenden-, Kreuz- und Schwanzwirbel erkennen.","Die Wirbelsäule der Vögel läßt stets die verschiedenen Abteilungen in Hals, Rücken, Lenden, Kreuz und Schwanzwirbel erkennen."
7,Du fängſt dir einen Müßiggänger heraus und unterſuchſt ihn: wahrhaftig — ein Männchen.,Du fängst dir einen Müßiggänger heraus und untersuchst ihn : wahrhaftig ein Männchen.
8,"Es ist freylich ein übler Umstand, wenn Jemand seine Individualität zum Maassstabe macht, nach welcher Jedermann sich müsse beurtheilen lassen.","Es ist freilich ein übler Umstand, wenn Jemand seine Individualität zum Maßstabe macht, nach welcher Jedermann sich müsse beurteilen lassen."
9,"Kohlensäure ist gleichfalls ein Narcoticum und verzögert die Zusammenballung des Protoplasma, wenn später kohlensaures Ammoniak gegeben wird.","Kohlensäure ist gleichfalls ein Narkotickum und verzögert die Zusammenballung des Protoplasma, wenn später kohlensaures Ammonik gegeben wird."


# Modern data (`deu_news_2020`)

In [12]:
df_modern.head(20)

,orig,norm_pred
0,"Ein einstelliger Rang war auch in den letzten Jahren vor dem Abstieg 2013 nicht dabei, aber das soll sich künftig ändern.\n","Ein einstelliger Rang war auch in den letzten Jahren vor dem Abstieg 2013 nicht dabei, aber das soll sich künftig ändern."
1,"Porsche überschlägt sich auf A38 – Fahrer verletzt Am Porsche entstand Totalschaden, die Schadenshöhe ist im hohen fünfstelligen Bereich.\n","Porsche überschlägt sich auf A3S Fahrer verletzt Am Porsche entstand Totalschaden, die Schadenshöhe ist im hohen fünfstelligen Bereich."
2,Für Wohnzwecke umgebaut worden ist das ehemalige Werkstattgebäude hinter dem Wohnhaus.\n,Für Wohnzwecke umgebaut worden ist das ehemalige Werkstattgebäude hinter dem Wohnhaus.
3,Selbst wenn die zu Grünen mutieren würden.\n,Selbst wenn die zu Grünen mutieren würden.
4,Oder ist das ne Werbung für UCI?\n,Oder ist das ne Werbung für UVI?
5,Das Informationsbedürfnis von Gemeinden und Veranstaltern ist sehr hoch.\n,Das Informationsbedürfnis von Gemeinden und Veranstaltern ist sehr hoch.
6,"Das System schlägt automatisch noch freie Zeitslots vor, aufgrund der großen Nachfrage werden ausschließlich 2-Stunden-Tickets angeboten.\n","Das System schlägt automatisch noch freie Zeitslots vor, aufgrund der großen Nachfrage werden ausschließlich 2 - Stunden - Tickets angeboten."
7,Es ist sein zweiter Toter allein in diesem Jahr.\n,Es ist sein zweiter Toter allein in diesem Jahr.
8,"Gegen Leipzig und Mainz hätte der Aufsteiger stark performt und gezeigt, dass sie auch im Oberhaus ein ernstzunehmender Gegner sei.\n","Gegen Leipzig und Mainz hätte der Aufsteiger stark performt und gezeigt, dass sie auch im Oberhaus ein ernstzunehmender Gegner sei."
9,Damit werden jährlich auch rund 5'700 Tonnen Phosphor dem Kreislauf entzogen und landen mit Asche und Schlacke unwiederbringlich auf Deponien.\n,Damit werden jährlich auch rund 5'700 Tonnen Phosphor dem Kreislauf entzogen und landen mit Asche und Schlacke unwiederbringlich auf Deponien.
